# Chapter 7 — First-Order Logic

**The Encyclopedia of Artificial Intelligence** — Perla Genovesi

This notebook contains the Python example from Chapter 7. Press the play button (▶) next to the code cell to run it: the result appears just below.

No installation is needed. The code uses only Python's standard library.

In [ ]:
# Datalog fragment: constants in lower case, variables with an initial capital
def is_variable(t):
    return isinstance(t, str) and t[:1].isupper()

def unify(x, y, theta):
    """Unification without function symbols: a substitution or None."""
    if theta is None:
        return None
    if x == y:
        return theta
    if is_variable(x):
        return unify_variable(x, y, theta)
    if is_variable(y):
        return unify_variable(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
        return theta
    return None

def unify_variable(v, t, theta):
    if v in theta:
        return unify(theta[v], t, theta)
    if is_variable(t) and t in theta:
        return unify(v, theta[t], theta)
    return {**theta, v: t}

def substitute(atom, theta):
    return tuple(theta.get(t, t) if is_variable(t) else t for t in atom)

def satisfy(premises, facts, theta):
    """Generate all substitutions that make the premises true."""
    if not premises:
        yield theta
        return
    for fact in facts:
        new = unify(premises[0], fact, theta)
        if new is not None:
            yield from satisfy(premises[1:], facts, new)

def forward(rules, facts):
    """Forward chaining with variables, up to the fixed point."""
    facts = set(facts)
    while True:
        new_facts = set()
        for premises, conclusion in rules:
            for theta in satisfy(premises, list(facts), {}):
                fact = substitute(conclusion, theta)
                if fact not in facts:
                    new_facts.add(fact)
        if not new_facts:
            return facts
        facts |= new_facts

def query(question, facts):
    answers = []
    for fact in sorted(facts):
        theta = unify(question, fact, {})
        if theta is not None:
            answers.append(theta)
    return answers

RULES = [
    ([("parent", "X", "Y")], ("ancestor", "X", "Y")),
    ([("parent", "X", "Y"), ("ancestor", "Y", "Z")], ("ancestor", "X", "Z")),
    ([("parent", "X", "Y"), ("parent", "Y", "Z")], ("grandparent", "X", "Z")),
]
FACTS = [("parent", "anna", "bruno"), ("parent", "bruno", "carla"),
         ("parent", "carla", "davide"), ("parent", "anna", "elena")]

print("unify(loves(X, Y), loves(anna, Z)):",
      unify(("loves", "X", "Y"), ("loves", "anna", "Z"), {}))
facts = forward(RULES, FACTS)
print("derived facts:", len(facts) - len(FACTS))
print("ancestor(anna, Who):",
      [t["Who"] for t in query(("ancestor", "anna", "Who"), facts)])
print("grandparent(Who, davide):",
      [t["Who"] for t in query(("grandparent", "Who", "davide"), facts)])

## Expected output

If everything works, you should see:

```
unify(loves(X, Y), loves(anna, Z)): {'X': 'anna', 'Y': 'Z'}
derived facts: 9
ancestor(anna, Who): ['bruno', 'carla', 'davide', 'elena']
grandparent(Who, davide): ['bruno']
```

Now try changing a number in the code and run it again to see what happens.